In [ ]:
from pathlib import Path
import subprocess
import sys
project_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(project_root / 'requirements.txt')])


In [ ]:
from pathlib import Path
import os
import random
import numpy as np
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / 'data'
RAW_DIR = DATA_DIR / 'raw'
PROCESSED_DIR = DATA_DIR / 'processed'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
MODELS_DIR = PROJECT_ROOT / 'models'
NOTEBOOKS_DIR = PROJECT_ROOT / 'notebooks'
print(f'Seed fixed at {SEED}')
print(f'Project root: {PROJECT_ROOT}')


# Notebook 06 ? Multi-Hazard Inference + Risk Mapping
## Builds per-hazard maps and combined risk layers


## Section 6.1 — Load CASA-Net

The inference notebook loads the final checkpoint and reuses the saved ablation summary so it can stand on its own. This keeps the scene-mapping stage self-contained and reproducible.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from rasterio.windows import Window
from tqdm.auto import tqdm

PROCESSED_SAR = PROCESSED_DIR / 'sar'
PROCESSED_TERRAIN = PROCESSED_DIR / 'terrain'
MAPS_DIR = OUTPUTS_DIR / 'maps'
FIGURES_DIR = OUTPUTS_DIR / 'figures'
REPORT_DIR = OUTPUTS_DIR / 'report'
MAPS_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class CPAG(nn.Module):
    def __init__(self, vv_channels, vh_channels, inter_channels):
        super().__init__(); self.Q = nn.Conv2d(vv_channels, inter_channels, 1); self.K = nn.Conv2d(vh_channels, inter_channels, 1); self.V = nn.Conv2d(vh_channels, vh_channels, 1); self.proj = nn.Conv2d(vh_channels, vv_channels, 1); self.scale = inter_channels ** -0.5
    def forward(self, f_vv, g_vh):
        if g_vh.shape[-2:] != f_vv.shape[-2:]:
            g_vh = F.interpolate(g_vh, size=f_vv.shape[-2:], mode='bilinear', align_corners=False)
        q = self.Q(f_vv).flatten(2); k = self.K(g_vh).flatten(2); v = self.V(g_vh).flatten(2)
        b, _, hw = q.shape; side = int(hw ** 0.5)
        attn = torch.softmax(torch.bmm(q.transpose(1, 2), k) * self.scale, dim=-1)
        return f_vv + self.proj(torch.bmm(v, attn.transpose(1, 2)).reshape(b, -1, side, side))

def apply_film(x, film_params):
    gamma, beta = film_params.chunk(2, dim=1); return gamma.unsqueeze(-1).unsqueeze(-1) * x + beta.unsqueeze(-1).unsqueeze(-1)

class HaorTerrainBranch(nn.Module):
    def __init__(self):
        super().__init__(); self.encoder = nn.Sequential(nn.Conv2d(4, 32, 3, padding=1), nn.ReLU(inplace=True), nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(inplace=True), nn.AdaptiveAvgPool2d(1)); self.film_generators = nn.ModuleList([nn.Linear(64, 2 * ch) for ch in (256, 128, 64, 32)])
    def forward(self, terrain_input):
        context = self.encoder(terrain_input).flatten(1); return [layer(context) for layer in self.film_generators]

class ResNet34Encoder(nn.Module):
    def __init__(self):
        super().__init__(); backbone = torchvision.models.resnet34(weights=None); backbone.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False); self.stem = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu); self.maxpool = backbone.maxpool; self.layer1 = backbone.layer1; self.layer2 = backbone.layer2; self.layer3 = backbone.layer3; self.layer4 = backbone.layer4
    def forward(self, x):
        x = self.stem(x); f1 = self.layer1(self.maxpool(x)); f2 = self.layer2(f1); f3 = self.layer3(f2); f4 = self.layer4(f3); return [f1, f2, f3, f4]

class MobileNetV3Encoder(nn.Module):
    def __init__(self):
        super().__init__(); backbone = torchvision.models.mobilenet_v3_small(weights=None); backbone.features[0][0] = nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1, bias=False); self.features = backbone.features
    def forward(self, x):
        outs = []
        for idx, layer in enumerate(self.features):
            x = layer(x)
            if idx in {1, 3, 6, 11}: outs.append(x)
        return outs

class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__(); self.conv1 = nn.Conv2d(in_channels + skip_channels, out_channels, 3, padding=1); self.bn1 = nn.BatchNorm2d(out_channels); self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1); self.bn2 = nn.BatchNorm2d(out_channels)
    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[-2:], mode='bilinear', align_corners=False); x = torch.cat([x, skip], dim=1); x = F.relu(self.bn1(self.conv1(x)), inplace=True); x = F.relu(self.bn2(self.conv2(x)), inplace=True); return x

class CASANet(nn.Module):
    def __init__(self):
        super().__init__(); self.vv_encoder = ResNet34Encoder(); self.vh_encoder = MobileNetV3Encoder(); self.cpag = nn.ModuleList([CPAG(64, 16, 32), CPAG(128, 24, 64), CPAG(256, 48, 128), CPAG(512, 96, 256)]); self.terrain_branch = HaorTerrainBranch(); self.block1 = DecoderBlock(512, 256, 256); self.block2 = DecoderBlock(256, 128, 128); self.block3 = DecoderBlock(128, 64, 64); self.final = nn.Sequential(nn.Conv2d(64, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.Dropout2d(0.3), nn.Conv2d(32, 1, 1))
    def forward(self, x_vv, x_vh, x_terrain):
        vv_feats = self.vv_encoder(x_vv); vh_feats = self.vh_encoder(x_vh); fused = [self.cpag[i](vv_feats[i], vh_feats[i]) for i in range(4)]; film = self.terrain_branch(x_terrain); x = self.block1(fused[-1], fused[-2]); x = apply_film(x, film[0]); x = self.block2(x, fused[-3]); x = apply_film(x, film[1]); x = self.block3(x, fused[-4]); x = apply_film(x, film[2]); x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False); x = self.final[:-1](x); x = apply_film(x, film[3]); return torch.sigmoid(self.final[-1](x))

@torch.no_grad()
def mc_inference(model, x_vv, x_vh, terrain, n_passes=10):
    was_training = model.training; model.train(); preds = torch.stack([model(x_vv, x_vh, terrain) for _ in range(n_passes)], dim=0); mean_pred = preds.mean(0); uncertainty = preds.std(0); model.train(was_training); return mean_pred, uncertainty

model = CASANet().to(DEVICE)
model.load_state_dict(torch.load(MODELS_DIR / 'casa_net_best.pth', map_location=DEVICE))
model.eval()
ablation_df = pd.read_csv(REPORT_DIR / 'ablation_results.csv')
display(ablation_df)


## Section 6.2 — Full Scene Inference Function

Whole-scene inference uses overlapping tiles and Gaussian-weighted blending to reduce patch-edge artifacts. MC Dropout provides the uncertainty layer that the infrastructure notebook will use later.


In [ ]:
def gaussian_kernel(size=256, sigma=48):
    coords = np.arange(size) - (size - 1) / 2
    xx, yy = np.meshgrid(coords, coords)
    kernel = np.exp(-(xx**2 + yy**2) / (2 * sigma**2))
    return kernel / kernel.max()

def read_patch(src, window, patch_size=256):
    arr = src.read(1, window=window, boundless=True, fill_value=0).astype(np.float32)
    out = np.zeros((patch_size, patch_size), dtype=np.float32)
    out[:arr.shape[0], :arr.shape[1]] = arr
    return out

def infer_full_scene(vv_tif, vh_tif, terrain_dict, model, patch_size=256, overlap=64, threshold=0.5, mc_passes=10):
    stride = patch_size - overlap
    kernel = gaussian_kernel(patch_size)
    with rasterio.open(vv_tif) as vv_src, rasterio.open(vh_tif) as vh_src:
        terrain_sources = {name: rasterio.open(path) for name, path in terrain_dict.items()}
        try:
            prob_accum = np.zeros((vv_src.height, vv_src.width), dtype=np.float32); unc_accum = np.zeros((vv_src.height, vv_src.width), dtype=np.float32); weight_accum = np.zeros((vv_src.height, vv_src.width), dtype=np.float32)
            for row_off in tqdm(range(0, vv_src.height, stride), desc=f'Inference {Path(vv_tif).stem}'):
                for col_off in range(0, vv_src.width, stride):
                    window = Window(col_off, row_off, patch_size, patch_size)
                    vv = read_patch(vv_src, window, patch_size); vh = read_patch(vh_src, window, patch_size); terrain = np.stack([read_patch(src, window, patch_size) for src in terrain_sources.values()], axis=0)
                    mean_pred, uncertainty = mc_inference(model, torch.tensor(vv[None, None, ...], device=DEVICE), torch.tensor(vh[None, None, ...], device=DEVICE), torch.tensor(terrain[None, ...], device=DEVICE), n_passes=mc_passes)
                    mean_np = mean_pred.cpu().numpy()[0, 0]; unc_np = uncertainty.cpu().numpy()[0, 0]
                    row_end = min(row_off + patch_size, vv_src.height); col_end = min(col_off + patch_size, vv_src.width); ks = kernel[:row_end - row_off, :col_end - col_off]
                    prob_accum[row_off:row_end, col_off:col_end] += mean_np[:row_end - row_off, :col_end - col_off] * ks
                    unc_accum[row_off:row_end, col_off:col_end] += unc_np[:row_end - row_off, :col_end - col_off] * ks
                    weight_accum[row_off:row_end, col_off:col_end] += ks
            prob_map = prob_accum / np.clip(weight_accum, 1e-6, None); unc_map = unc_accum / np.clip(weight_accum, 1e-6, None); flood_map = (prob_map >= threshold).astype(np.uint8); profile = vv_src.profile.copy(); profile.update(count=1)
        finally:
            for src in terrain_sources.values(): src.close()
    return flood_map, prob_map.astype(np.float32), unc_map.astype(np.float32), profile


## Section 6.3 — Batch Inference (All 4 Dates)

Exporting binary, probability, and uncertainty rasters for each date gives us both operational map products and reusable inputs for the overlay and threshold notebooks.


In [ ]:
terrain_dict = {'slope': PROCESSED_TERRAIN / 'slope_sylhet.tif', 'twi': PROCESSED_TERRAIN / 'twi_sylhet.tif', 'jrc': PROCESSED_TERRAIN / 'jrc_water_sylhet.tif', 'hand': PROCESSED_TERRAIN / 'hand_sylhet.tif'}
rows = []
for vv_tif in sorted(PROCESSED_SAR.glob('S1_VV_*.tif')):
    date_token = vv_tif.stem.split('_')[-1]
    vh_tif = PROCESSED_SAR / f'S1_VH_{date_token}.tif'
    if not vh_tif.exists():
        continue
    flood_map, prob_map, unc_map, profile = infer_full_scene(vv_tif, vh_tif, terrain_dict, model, patch_size=256, overlap=64, threshold=0.5, mc_passes=10)
    for out_path, arr, dtype in [(MAPS_DIR / f'flood_binary_{date_token}.tif', flood_map.astype(np.uint8), 'uint8'), (MAPS_DIR / f'flood_prob_{date_token}.tif', prob_map.astype(np.float32), 'float32'), (MAPS_DIR / f'flood_uncertainty_{date_token}.tif', unc_map.astype(np.float32), 'float32')]:
        out_profile = profile.copy(); out_profile.update(dtype=dtype)
        with rasterio.open(out_path, 'w', **out_profile) as dst: dst.write(arr, 1)
    pixel_area_km2 = abs(profile['transform'].a * profile['transform'].e) / 1e6
    rows.append({'date': date_token, 'year': date_token[:4], 'binary_path': str(MAPS_DIR / f'flood_binary_{date_token}.tif'), 'prob_path': str(MAPS_DIR / f'flood_prob_{date_token}.tif'), 'uncertainty_path': str(MAPS_DIR / f'flood_uncertainty_{date_token}.tif'), 'flooded_area_km2': flood_map.sum() * pixel_area_km2})
inference_df = pd.DataFrame(rows).sort_values('date')
inference_df.to_csv(REPORT_DIR / 'flood_area_by_date.csv', index=False)
display(inference_df.head(20))


## Section 6.4 — Accuracy Assessment

Scene-level comparison against UNOSAT and the UNICEF June benchmark complements the patch-level metrics from Notebook 05. It also creates one of the required evaluation diagrams for the final competition package.


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
assessment_rows = []
for _, inf_row in inference_df.iterrows():
    date_token = inf_row['date']
    candidate_masks = sorted((PROCESSED_DIR / 'masks').glob(f'*{date_token}.tif'))
    if not candidate_masks:
        continue
    with rasterio.open(candidate_masks[0]) as ref_src, rasterio.open(inf_row['binary_path']) as pred_src:
        ref = ref_src.read(1).astype(np.uint8); pred = pred_src.read(1).astype(np.uint8); pixel_area_km2 = abs(ref_src.transform.a * ref_src.transform.e) / 1e6
    assessment_rows.append({'date': date_token, 'Accuracy': accuracy_score(ref.ravel(), pred.ravel()), 'IoU': np.logical_and(ref == 1, pred == 1).sum() / max(np.logical_or(ref == 1, pred == 1).sum(), 1), 'F1': f1_score(ref.ravel(), pred.ravel(), zero_division=0), 'Precision': precision_score(ref.ravel(), pred.ravel(), zero_division=0), 'Recall': recall_score(ref.ravel(), pred.ravel(), zero_division=0), 'Predicted_km2': pred.sum() * pixel_area_km2, 'Reference_km2': ref.sum() * pixel_area_km2})
assessment_df = pd.DataFrame(assessment_rows)
assessment_df.to_csv(REPORT_DIR / 'scene_accuracy_assessment.csv', index=False)
display(assessment_df)
if not assessment_df.empty:
    row = assessment_df.iloc[0]
    fig, ax = plt.subplots(figsize=(8, 5), dpi=300)
    ax.bar(['Predicted', 'Reference'], [row['Predicted_km2'], row['Reference_km2']], color=['#264653', '#2a9d8f'])
    ax.set_ylabel('Flooded area (km²)'); ax.set_title(f"Extent comparison {row['date']}")
    plt.tight_layout(); plt.savefig(FIGURES_DIR / 'scene_extent_validation.png', dpi=300, bbox_inches='tight'); plt.show(); plt.close(fig)


## Section 6.5 — Flood Progression Visualization

The progression figure summarizes how flood extent changes across the four target dates. It is one of the clearest visuals for the narrative side of the submission deck.


In [ ]:
plot_df = inference_df.copy().sort_values('date').head(4)
fig, axes = plt.subplots(2, 2, figsize=(14, 10), dpi=300)
for ax, (_, row) in zip(axes.ravel(), plot_df.iterrows()):
    date_token = row['date']
    with rasterio.open(PROCESSED_SAR / f'S1_VV_{date_token}.tif') as vv_src, rasterio.open(MAPS_DIR / f'flood_binary_{date_token}.tif') as flood_src:
        vv = vv_src.read(1); flood = flood_src.read(1); extent = [vv_src.bounds.left, vv_src.bounds.right, vv_src.bounds.bottom, vv_src.bounds.top]
    ax.imshow(vv, cmap='gray', vmin=-25, vmax=0, extent=extent); ax.imshow(np.ma.masked_where(flood == 0, flood), cmap='Blues', alpha=0.55, extent=extent); ax.set_title(date_token)
plt.tight_layout(); plt.savefig(FIGURES_DIR / 'flood_progression.png', dpi=300, bbox_inches='tight'); plt.show(); plt.close(fig)


## Section 6.6 — Uncertainty Map Visualization

The uncertainty maps identify ambiguous flood zones, especially around haor boundaries. Those high-uncertainty areas are explicitly passed into the infrastructure overlay notebook.


In [ ]:
focus_dates = inference_df.sort_values('flooded_area_km2', ascending=False)['date'].head(2).tolist()
for date_token in focus_dates:
    with rasterio.open(PROCESSED_SAR / f'S1_VV_{date_token}.tif') as vv_src, rasterio.open(MAPS_DIR / f'flood_prob_{date_token}.tif') as prob_src, rasterio.open(MAPS_DIR / f'flood_uncertainty_{date_token}.tif') as unc_src:
        vv = vv_src.read(1); prob = prob_src.read(1); unc = unc_src.read(1); extent = [vv_src.bounds.left, vv_src.bounds.right, vv_src.bounds.bottom, vv_src.bounds.top]
    fig, axes = plt.subplots(1, 3, figsize=(14, 4), dpi=300)
    axes[0].imshow(prob, cmap='Blues', vmin=0, vmax=1, extent=extent); axes[0].set_title(f"Probability {date_token}")
    axes[1].imshow(unc, cmap='magma', extent=extent); axes[1].set_title(f"Uncertainty {date_token}")
    axes[2].imshow(vv, cmap='gray', vmin=-25, vmax=0, extent=extent); axes[2].imshow(np.ma.masked_where(unc <= 0.3, unc), cmap='autumn', alpha=0.6, extent=extent); axes[2].set_title('High-uncertainty zones > 0.30')
    plt.tight_layout(); plt.savefig(FIGURES_DIR / f"uncertainty_maps_{date_token}.png", dpi=300, bbox_inches='tight'); plt.show(); plt.close(fig)


## Section 6.7 — Flood Area Timeline Chart

The flooded-area timeline is a compact scene-level evaluation chart and also feeds directly into the anticipatory-action threshold notebook.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4), dpi=300)
ax.bar(inference_df['date'], inference_df['flooded_area_km2'], color='#1d3557')
ax.set_title('Flooded area timeline'); ax.set_xlabel('Date'); ax.set_ylabel('Flooded area (km²)')
for idx, row in inference_df.iterrows():
    ax.text(idx, row['flooded_area_km2'], f"{row['flooded_area_km2']:.0f}", ha='center', va='bottom')
plt.tight_layout(); plt.savefig(FIGURES_DIR / 'flood_area_timeline.png', dpi=300, bbox_inches='tight'); plt.show(); plt.close(fig)


<!-- MULTI-HAZARD EXTENSION GENERATED -->
## Section 6.8 ? Multi-Hazard Inference and Combined Risk
The flood notebook is extended here into a **per-hazard inference planner**. Each hazard can emit probability, binary, and uncertainty layers, then all hazard probabilities can be fused into a combined risk surface.


In [ ]:
# MULTI-HAZARD EXTENSION GENERATED
import pandas as pd
from analysis.multi_hazard_support import load_hazard_catalog, ordered_hazard_ids, build_combined_risk_layer

ROOT = Path(r'f:\MAPATHON\sylhet_flood_2024')
hazard_catalog = load_hazard_catalog(ROOT / 'config' / 'hazard_catalog.json')
hazard_names = ordered_hazard_ids(hazard_catalog)

def infer_multi_hazard_scene(vv_tif, vh_tif, terrain_dict, model, hazard_names=hazard_names):
    """Wrapper for future multi-hazard scene inference. If `model` returns a dict, each hazard is exported separately."""
    map_plan = []
    for hazard in hazard_names:
        date_tag = Path(vv_tif).stem.split('_')[-1] if vv_tif else 'YYYYMMDD'
        map_plan.extend([
            {'hazard': hazard, 'map_type': 'probability', 'output_path': str(ROOT / 'outputs' / 'maps' / hazard / f"{hazard}_prob_{date_tag}.tif")},
            {'hazard': hazard, 'map_type': 'binary', 'output_path': str(ROOT / 'outputs' / 'maps' / hazard / f"{hazard}_binary_{date_tag}.tif")},
            {'hazard': hazard, 'map_type': 'uncertainty', 'output_path': str(ROOT / 'outputs' / 'maps' / hazard / f"{hazard}_uncertainty_{date_tag}.tif")},
        ])
    return pd.DataFrame(map_plan)

combined_risk_example = build_combined_risk_layer(
    {hazard: np.zeros((8, 8), dtype=np.float32) for hazard in hazard_names},
    hazard_weights=hazard_catalog['combined_risk']['default_hazard_weights'],
)
multi_hazard_map_inventory = infer_multi_hazard_scene('S1_VV_20240619.tif', 'S1_VH_20240619.tif', {}, None)
multi_hazard_map_inventory.to_csv(ROOT / 'outputs' / 'report' / 'multi_hazard_map_inventory.csv', index=False)
multi_hazard_map_inventory.head()
